# Adventure Works – End-to-End ETL Pipeline

**Objective:** Extract data from Excel, explore it, clean it, validate it,
and load the cleaned tables into MySQL.

**Workbook:** Adventure Works Project Excel File.xlsx

**Tables used:** SALES, PRODUCT, Dimcustomer, DimSalesTerritory, DimDate

Run each cell from top to bottom in Jupyter Notebook.

## 1. Install required libraries
Run this cell once if the libraries are not already installed.

In [ ]:
%pip install pandas openpyxl sqlalchemy pymysql

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 3. Set file path and MySQL configuration

Update EXCEL_FILE with the location of your workbook.
Update MYSQL_PASSWORD with your MySQL password.
The database will be created automatically if it does not exist.

In [ ]:
EXCEL_FILE = Path(r"C:\Users\YourName\Downloads\Adventure Works Project Excel File.xlsx")

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "YOUR_MYSQL_PASSWORD"
MYSQL_DATABASE = "adventure_works"

# Check whether the Excel file exists
if not EXCEL_FILE.exists():
    raise FileNotFoundError(
        f"Excel file not found: {EXCEL_FILE}\n"
        "Please update EXCEL_FILE with the correct workbook path."
    )

print("Excel file found:", EXCEL_FILE)

## 4. DATA EXTRACTION
Read the required sheets from the Excel workbook.

In [ ]:
required_sheets = [
    "SALES",
    "PRODUCT",
    "Dimcustomer",
    "DimSalesTerritory",
    "DimDate"
]

# Read all required sheets into a dictionary of DataFrames
raw_data = pd.read_excel(
    EXCEL_FILE,
    sheet_name=required_sheets,
    engine="openpyxl"
)

print("Data extraction completed successfully.")

for sheet_name, data in raw_data.items():
    print(f"{sheet_name}: {data.shape[0]} rows and {data.shape[1]} columns")

## 5. DATA EXPLORATION
Inspect the extracted datasets before making changes.

In [ ]:
for sheet_name, data in raw_data.items():
    print("\n" + "=" * 70)
    print("TABLE:", sheet_name)
    print("=" * 70)

    print("\nFirst 5 records:")
    display(data.head())

    print("\nDataset information:")
    data.info()

    print("\nDataset dimensions:")
    print(data.shape)

    print("\nColumn names:")
    print(data.columns.tolist())

    print("\nMissing values:")
    print(data.isnull().sum())

    print("\nDuplicate records:")
    print(data.duplicated().sum())

### Summary statistics for numeric columns

In [ ]:
for sheet_name, data in raw_data.items():
    print(f"\nSummary statistics: {sheet_name}")
    display(data.describe(include="all").T)

## 6. DATA CLEANING
We will:
- Make column names lowercase and replace spaces with underscores.
- Remove leading/trailing spaces from text values.
- Remove completely duplicated records.
- Convert date columns into proper datetime values where applicable.
- Handle missing values only where it is safe to do so.

Important: We will not fill every missing value with zero. A missing value
may mean "not applicable" rather than zero.

In [ ]:
cleaned_data = {}

for sheet_name, original_df in raw_data.items():

    # Work on a copy so the original extracted data remains unchanged
    df = original_df.copy()

    # 1. Standardise column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # 2. Clean text columns
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        df[column] = df[column].apply(
            lambda value: value.strip() if isinstance(value, str) else value
        )

        # Convert empty strings to missing values
        df[column] = df[column].replace("", np.nan)

    # 3. Remove fully duplicated rows
    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates()

    print(
        f"{sheet_name}: removed {duplicate_count} fully duplicated rows"
    )

    cleaned_data[sheet_name] = df

print("\nBasic cleaning completed.")

### 6.1 Clean the SALES table
Sales transaction keys and measures should be checked carefully.
Rows missing essential identifiers are removed. Optional fields are retained
as missing rather than filled with invented values.

In [ ]:
sales_df = cleaned_data["SALES"].copy()

# Convert sales measures to numeric values
sales_numeric_columns = [
    "order_quantity",
    "unit_price",
    "extended_amount",
    "unit_price_discount_pct",
    "discount_amount",
    "product_standard_cost",
    "tax_amt",
    "freight"
]

for column in sales_numeric_columns:
    if column in sales_df.columns:
        sales_df[column] = pd.to_numeric(sales_df[column], errors="coerce")

# Convert date columns to datetime
sales_date_columns = ["order_date", "due_date", "ship_date"]

for column in sales_date_columns:
    if column in sales_df.columns:
        sales_df[column] = pd.to_datetime(
            sales_df[column], errors="coerce"
        )

# Essential identifiers for a sales transaction
sales_required_columns = [
    "product_key",
    "customer_key",
    "sales_order_number",
    "sales_order_line_number"
]

existing_required_columns = [
    col for col in sales_required_columns if col in sales_df.columns
]

before_rows = len(sales_df)

if existing_required_columns:
    sales_df = sales_df.dropna(subset=existing_required_columns)

print("Sales rows before essential-key check:", before_rows)
print("Sales rows after essential-key check:", len(sales_df))

# Check for invalid negative quantities or prices
if "order_quantity" in sales_df.columns:
    invalid_quantity = (sales_df["order_quantity"] < 0).sum()
    print("Rows with negative order quantity:", invalid_quantity)

if "unit_price" in sales_df.columns:
    invalid_price = (sales_df["unit_price"] < 0).sum()
    print("Rows with negative unit price:", invalid_price)

cleaned_data["SALES"] = sales_df

### 6.2 Clean the PRODUCT table

In [ ]:
product_df = cleaned_data["PRODUCT"].copy()

product_numeric_columns = [
    "product_key",
    "product_category_key",
    "product_subcategory_key",
    "standard_cost",
    "list_price",
    "dealer_price",
    "safety_stock_level",
    "reorder_point",
    "weight",
    "days_to_manufacture"
]

for column in product_numeric_columns:
    if column in product_df.columns:
        product_df[column] = pd.to_numeric(
            product_df[column], errors="coerce"
        )

# Product key is essential for identifying a product
if "product_key" in product_df.columns:
    product_df = product_df.dropna(subset=["product_key"])

cleaned_data["PRODUCT"] = product_df

### 6.3 Clean the CUSTOMER table

In [ ]:
customer_df = cleaned_data["Dimcustomer"].copy()

customer_numeric_columns = [
    "customer_key",
    "geography_key",
    "yearly_income",
    "total_children",
    "number_children_at_home",
    "number_cars_owned"
]

for column in customer_numeric_columns:
    if column in customer_df.columns:
        customer_df[column] = pd.to_numeric(
            customer_df[column], errors="coerce"
        )

customer_date_columns = ["birthdate", "date_first_purchase"]

for column in customer_date_columns:
    if column in customer_df.columns:
        customer_df[column] = pd.to_datetime(
            customer_df[column], errors="coerce"
        )

if "customer_key" in customer_df.columns:
    customer_df = customer_df.dropna(subset=["customer_key"])

cleaned_data["Dimcustomer"] = customer_df

### 6.4 Clean the SALES TERRITORY table

In [ ]:
territory_df = cleaned_data["DimSalesTerritory"].copy()

territory_numeric_columns = [
    "sales_territory_key",
    "sales_territory_alternate_key"
]

for column in territory_numeric_columns:
    if column in territory_df.columns:
        territory_df[column] = pd.to_numeric(
            territory_df[column], errors="coerce"
        )

if "sales_territory_key" in territory_df.columns:
    territory_df = territory_df.dropna(
        subset=["sales_territory_key"]
    )

cleaned_data["DimSalesTerritory"] = territory_df

### 6.5 Clean the DATE dimension table

In [ ]:
date_df = cleaned_data["DimDate"].copy()

date_numeric_columns = [
    "datekey",
    "daynumberofweek",
    "daynumberofmonth",
    "daynumberofyear",
    "weeknumberofyear",
    "monthnumberofyear",
    "calendarquarter",
    "calendaryear",
    "calendarsemester",
    "fiscalquarter",
    "fiscalyear",
    "fiscalsemester"
]

for column in date_numeric_columns:
    if column in date_df.columns:
        date_df[column] = pd.to_numeric(
            date_df[column], errors="coerce"
        )

if "fulldatealternatekey" in date_df.columns:
    date_df["fulldatealternatekey"] = pd.to_datetime(
        date_df["fulldatealternatekey"], errors="coerce"
    )

if "datekey" in date_df.columns:
    date_df = date_df.dropna(subset=["datekey"])

cleaned_data["DimDate"] = date_df

print("Table-specific cleaning completed.")

## 7. DATA VALIDATION
Validate each cleaned table:
1. Check missing values.
2. Check duplicate records.
3. Check dataset dimensions.

Missing values in optional columns may be valid. Review them rather than
automatically deleting or replacing them.

In [ ]:
for table_name, df in cleaned_data.items():

    print("\n" + "=" * 70)
    print("FINAL VALIDATION:", table_name)
    print("=" * 70)

    # Check missing values
    print("\nFinal Missing Values:")
    print(df.isnull().sum())

    # Check duplicate records
    print("\nDuplicate Records:")
    print(df.duplicated().sum())

    # Check dataset dimensions
    print("\nFinal Dataset Shape:")
    print(df.shape)

### 7.1 Check primary-key duplicates
A primary key should uniquely identify each row in its dimension table.

In [ ]:
primary_keys = {
    "PRODUCT": "product_key",
    "Dimcustomer": "customer_key",
    "DimSalesTerritory": "sales_territory_key",
    "DimDate": "datekey"
}

for table_name, key_column in primary_keys.items():
    df = cleaned_data[table_name]

    if key_column in df.columns:
        print(
            f"{table_name} duplicate {key_column} values:",
            df[key_column].duplicated().sum()
        )

### 7.2 Validate sales relationships
Check whether sales records reference product, customer and territory keys
that exist in their respective dimension tables.

In [ ]:
sales_df = cleaned_data["SALES"]

relationship_checks = {
    "product_key": ("PRODUCT", "product_key"),
    "customer_key": ("Dimcustomer", "customer_key"),
    "sales_territory_key": (
        "DimSalesTerritory",
        "sales_territory_key"
    )
}

for sales_column, (dimension_table, dimension_column) in relationship_checks.items():

    if (
        sales_column in sales_df.columns
        and dimension_column in cleaned_data[dimension_table].columns
    ):
        valid_keys = set(
            cleaned_data[dimension_table][dimension_column].dropna()
        )

        unmatched = ~sales_df[sales_column].isin(valid_keys)

        print(
            f"Unmatched {sales_column} values:",
            unmatched.sum()
        )

## 8. LOAD CLEANED DATA INTO MYSQL
This section creates the database if it does not exist and loads each
cleaned DataFrame into a separate MySQL table.

Note: if_exists="replace" replaces the table each time this notebook runs.
Use "append" only when you intentionally want to add records.

In [ ]:
# Create a connection to the MySQL server (without selecting a database)
server_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT
)

server_engine = create_engine(server_url)

# Create the database if it does not already exist
with server_engine.connect() as connection:
    connection.execute(
        text(
            f"CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}` "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
        )
    )

print(f"MySQL database '{MYSQL_DATABASE}' is ready.")

### 8.1 Connect to the Adventure Works database

In [ ]:
database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE
)

engine = create_engine(database_url)

# Test the connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE()"))
    print("Connected to database:", result.scalar())

### 8.2 Load the cleaned tables

In [ ]:
table_name_mapping = {
    "SALES": "sales",
    "PRODUCT": "product",
    "Dimcustomer": "dim_customer",
    "DimSalesTerritory": "dim_sales_territory",
    "DimDate": "dim_date"
}

for source_name, df in cleaned_data.items():

    mysql_table = table_name_mapping[source_name]

    df.to_sql(
        name=mysql_table,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000,
        method="multi"
    )

    print(
        f"Loaded {len(df):,} rows into MySQL table '{mysql_table}'."
    )

## 9. VERIFY DATA LOADED INTO MYSQL
Read each table back from MySQL and compare its row count with the
cleaned DataFrame.

In [ ]:
for source_name, mysql_table in table_name_mapping.items():

    mysql_df = pd.read_sql(
        text(f"SELECT * FROM `{mysql_table}`"),
        con=engine
    )

    expected_rows = len(cleaned_data[source_name])
    actual_rows = len(mysql_df)

    print(f"\nTable: {mysql_table}")
    print("Expected rows:", expected_rows)
    print("Rows in MySQL:", actual_rows)
    print("Load successful:", expected_rows == actual_rows)

## 10. FINAL MYSQL TABLE CHECK

In [ ]:
with engine.connect() as connection:

    for mysql_table in table_name_mapping.values():

        count = connection.execute(
            text(f"SELECT COUNT(*) FROM `{mysql_table}`")
        ).scalar()

        print(f"{mysql_table}: {count:,} records")

## 11. Close database connections

In [ ]:
engine.dispose()
server_engine.dispose()

print("ETL pipeline completed successfully.")

## ETL PIPELINE SUMMARY

**Extract:** Read SALES, PRODUCT, customer, territory and date sheets from Excel.

**Explore:** Inspected sample rows, data types, dimensions, statistics,
missing values and duplicates.

**Transform:** Standardised column names, trimmed text, removed duplicate
rows, converted data types and removed records missing essential keys.

**Validate:** Checked missing values, duplicates, dimensions, primary keys
and sales-to-dimension relationships.

**Load:** Loaded the cleaned tables into MySQL and verified row counts.